In [2]:
import torch.optim as optim
import torch.nn.functional as F
import torch.nn as nn
import torch
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [3]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

X_train_tensor = torch.from_numpy(X_train_scaled).float()
X_test_tensor = torch.from_numpy(X_test_scaled).float()
y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1)
y_test_tensor = torch.from_numpy(y_test).float().unsqueeze(1)

In [4]:
X_train_tensor.shape

torch.Size([426, 30])

In [22]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [27]:
class BcNet (nn.Module):
    def __init__(self):
        super(BcNet, self).__init__()

        self.layer1 = nn.Linear(30, 50)
        self.activation = nn.Linear(50, 15)
        self.layer2 = nn.Linear(15, 1)
    
    def forward(self, x):
        x = F.gelu(self.layer1(x))
        x = F.relu(self.activation(x))
        x = F.sigmoid(self.layer2(x))

        return x

In [28]:
model = BcNet()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)


In [47]:
epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        y_hat = model(X_batch)
        loss = criterion(y_hat, y_batch)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()
    print(f"in epoch({epoch+1}), loss: {running_loss}")

in epoch(1), loss: 0.003980383434190475
in epoch(2), loss: 0.003939010427038098
in epoch(3), loss: 0.0039019112926581556
in epoch(4), loss: 0.0038535792615306264
in epoch(5), loss: 0.003820702271711831
in epoch(6), loss: 0.003792278864695603
in epoch(7), loss: 0.0037510644401131865
in epoch(8), loss: 0.0037315107329618513
in epoch(9), loss: 0.003695439680848442
in epoch(10), loss: 0.007614027930742395
in epoch(11), loss: 0.0036023953497715754
in epoch(12), loss: 0.003572140270506452
in epoch(13), loss: 0.003527008122262032
in epoch(14), loss: 0.00348784319287887
in epoch(15), loss: 0.0034527843987576983
in epoch(16), loss: 0.003427125291069366
in epoch(17), loss: 0.0033944877780032606
in epoch(18), loss: 0.003361800190187836
in epoch(19), loss: 0.003333412009240533
in epoch(20), loss: 0.0033032313828584847


In [35]:
with torch.no_grad():
    model.eval()

    preds = model(X_test_tensor)
    loss = criterion(preds, y_test_tensor).item()

    accuracy = ((preds >= 0.5) == y_test_tensor).float().mean().item()

In [36]:
accuracy

0.9720279574394226